<a href="https://colab.research.google.com/github/AdityapIISERB/Bacterial-Gene-Network-inference/blob/main/data_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile generate_test_data.py
import numpy as np
import pandas as pd
import os

np.random.seed(42)
os.makedirs("data", exist_ok=True)

N_GENES = 20
TIMEPOINTS = [0, 0.25, 1, 4, 8, 24, 48]
CONDITIONS = ["control", "antibiotic"]
gene_ids = [f"gene_{i:04d}" for i in range(N_GENES)]
samples, sample_records = [], []

for cond in CONDITIONS:
    base_level = np.random.lognormal(mean=5, sigma=1.5, size=N_GENES)
    for tp in TIMEPOINTS:
        shift = 1 + 0.3 * np.log1p(tp) * np.random.choice([-1, 1], size=N_GENES, p=[0.3, 0.7]) if cond == "antibiotic" else 1 + 0.05 * np.random.randn(N_GENES)
        for rep in [1, 2]:
            sample_name = f"{cond}_T{tp}_rep{rep}"
            mean_expr = np.clip(base_level * shift, 0.1, None)
            counts = np.random.negative_binomial(n=5, p=5 / (5 + mean_expr))
            samples.append(pd.Series(counts, index=gene_ids, name=sample_name))
            sample_records.append({"sample_id": sample_name, "timepoint": tp, "condition": cond, "replicate": rep})

counts_df = pd.concat(samples, axis=1)
# Adjusted the number of genes to modify to be consistent with N_GENES
num_genes_to_modify = min(30, N_GENES)
counts_df.loc[gene_ids[-num_genes_to_modify:]] = np.random.poisson(0.3, size=(num_genes_to_modify, counts_df.shape[1]))

counts_df["antibiotic_T8_rep2"] = (counts_df["antibiotic_T8_rep2"] * 0.05).astype(int)
counts_df["control_T24_rep2"] = np.random.negative_binomial(n=2, p=0.3, size=counts_df.shape[0])

counts_df.to_csv("data/raw_counts.csv")
pd.DataFrame(sample_records).to_csv("data/sample_info.csv", index=False)
print("✓ Synthetic dataset generated in 'data/' directory.")

Overwriting generate_test_data.py


In [ ]:
!python generate_test_data.py

✓ Synthetic dataset generated in 'data/' directory.
